## START

In [1]:
# Load the data exactly as in homework 2:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

# This gives 72 pages.

In [2]:
len(documents) # 72

72

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [3]:
# Generating ground truth
# To evaluate search, we need a dataset of questions where we know which document is the correct answer. This is the ground truth.

# We generate it the same way as in the module. For each lesson page, we ask an LLM to write 5 questions that are answered by that page. 
# Each question is then labeled with the page it came from.

# We use the same structured-output approach as in the module - the same Questions model and the llm_structured helper from evaluation_utils.py.

# The module's instructions generate questions from a FAQ record, so we adapt them for a lesson page:

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


## Q1 -- 2

In [9]:
# We ask for different wording from the page on purpose. Real users don't phrase their questions the way the lesson does, 
# and copying the text would make the evaluation too easy.

# For each page, build a JSON user prompt from its filename and content, then call llm_structured with the Questions model. 
# Turn each returned question into a record labeled with the page's filename. 
# The call also returns the token usage, the same as in the lessons.

# Q1. Generating questions
# Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

# 01-agentic-rag/lessons/01-intro.md
# 01-agentic-rag/lessons/02-environment.md
# 01-agentic-rag/lessons/03-rag.md

# Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

# What's the average number of input tokens across these 3 calls?

# 140
# 1400 == average input tokens =  1020
# 14000
# 140000

# These numbers vary between runs, even with the same model, so pick the closest option. 
# A different provider or model may land further apart, but the input tokens stay in the same 
# order of magnitude - the prompt we send is the same.

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

doc = documents[0]
# print(doc["id"])
# print(doc["content"])
# print(doc["filename"])

In [10]:

# Call the LLM for one document:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Prepare the document as JSON:

import json

user_prompt = json.dumps(doc)
# Create the messages:

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

# The parsed object is available in response.output_parsed:

result = response.output_parsed

print(result)
# We can access the list directly:

print(result.questions)
# You should see 5 questions that relate to the first FAQ document.

questions=['What problem does retrieval-augmented generation solve for language models?', 'Why does this course build the RAG system in plain Python instead of using a framework right away?', 'What are the main limits of an LLM that make RAG useful?', 'What will the first part of this module build for the course FAQ bot?', 'How is the second part of the module different from the first one?']
['What problem does retrieval-augmented generation solve for language models?', 'Why does this course build the RAG system in plain Python instead of using a framework right away?', 'What are the main limits of an LLM that make RAG useful?', 'What will the first part of this module build for the course FAQ bot?', 'How is the second part of the module different from the first one?']


In [11]:
for q in result.questions:
    print(q)

What problem does retrieval-augmented generation solve for language models?
Why does this course build the RAG system in plain Python instead of using a framework right away?
What are the main limits of an LLM that make RAG useful?
What will the first part of this module build for the course FAQ bot?
How is the second part of the module different from the first one?


In [17]:
# llm_structured: calls the OpenAI API with structured output
# llm_structured_retry: retries structured-output calls when a request fails
# calc_price: calculates the price from token usage
# calc_total_price: calculates the total price from multiple usage objects
# map_progress: runs work in parallel and tracks progress. We'll use it in the next lesson.
# Import the structured-output helper:

from evaluation_utils import llm_structured
# Use it on the same document:

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

# Import the price helper:

from evaluation_utils import calc_price
# Calculate the cost of this call:

cost = calc_price(usage)

cost, usage, usage.input_tokens
print('usage.input_tokens = ' , usage.input_tokens)

['What problem does retrieval-augmented generation solve for an LLM?', 'Why does the course use plain Python instead of a framework at the start?', 'What are the main limits of an LLM that this lesson points out?', 'What kind of example system will this module build from the FAQ data?', 'What will be covered in the first part versus the second part of the module?']
usage.input_tokens =  1020


In [22]:
q1_docs = documents[:3]
its = 0 # sum of input tokens

for q in q1_docs:
    print(q['filename'])
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    print(result.questions)
    print('usage.input_tokens = ' , usage.input_tokens)
    its += usage.input_tokens

average = its/3
print('average input tokens = ', average)

01-agentic-rag/lessons/01-intro.md
['What is the main goal of this module, and what kind of system are we building?', 'Why does the lesson say to treat an LLM as a black box instead of studying how it works inside?', 'What problems with LLMs does retrieval-augmented generation help solve?', 'How does RAG help when the model needs information it did not learn during training?', 'What are the two parts of the module, and what changes in the second part?']
usage.input_tokens =  1020
01-agentic-rag/lessons/02-environment.md
["What is Retrieval-Augmented Generation, and how does it help when a model doesn't know the answer from training data alone?", 'Why does this lesson build the RAG system in plain Python instead of using a framework right away?', 'What are the main limits of large language models that make RAG useful?', 'What will the first part of this module cover before the pipeline becomes agentic?', 'What is the course project in this module, and what kind of questions will the sys

## Q2 -- 